In [1]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from typing import Dict

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector
from src.llm_agent.agent_model import SYSTEM_PROMPT

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

DATASET_PATH = '../data/Augment_DiaASQ.json'

In [2]:
class OpenAIAgent:

    def __init__(self, api_key, model: str = 'gpt-4o-mini') -> None:
        self.model = model
        self.client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", api_key))
        self.system_prompt = SYSTEM_PROMPT

    def generate(self, user_prompt: str, assistant_prompt: str = None, 
                 system_prompt: str = None, gen_strategy: Dict = None) -> str:
        """Метод для генерации ответов на текстовые запросы с помощью llm-агента.

        Args:
            user_prompt (str): Запрос для llm-агента.
            assistant_prompt (str, optional): Дополнительная к user_prompt-запросу информация, 
                                                которая может быть использована llm-агентом при генерации ответа. Defaults to None.
            gen_strategy (Dict, optional): Стретегия генерации текстовой последовательности для llm-агента. Defaults to None.

        Returns:
            str: Текстовая последовательность, сгенерированная llm-агентом.
        """
        messages = [
            {"role": "system", "content": system_prompt if system_prompt is not None else self.system_prompt},
            {"role": "user","content": user_prompt}]

        if assistant_prompt is not None:
            messages.insert(1, {"role": "assistant", "content": assistant_prompt})

        completion = self.client.chat.completions.create(
            model=self.model, messages=messages)

        return completion.choices[0].message.content

In [3]:
API_KEY = "<SECRET>"
agent = OpenAIAgent(api_key=API_KEY)

In [9]:
agent.generate("Сколько будет 2+2?")

'2 + 2 будет 4.'

### Extract

In [4]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [5]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))

In [6]:
len(raw_texts)

3483

In [4]:
#agent = AgentConnector.open()
#agent.generate("Сколько будет 2 + 2?")

In [7]:
extractor = LLMExtractor(agent_conn=agent)

In [8]:
extracted_triplets = []

In [9]:
for i in tqdm(range(len(raw_texts))):
    out = extractor.extract(raw_texts[i])
    extracted_triplets.append(out)

  0%|          | 9/3483 [00:56<6:07:41,  6.35s/it]

In [ ]:
with open("tmp_extracted_openai_gpt4omini_triplets.json", 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(extracted_triplets, ensure_ascii=False))

### Update

In [ ]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", default_db="diaasq"),
    embeddings_db=EmbeddingsDatabaseConnection()
)

In [ ]:
ids = kg_model.graph_db.create_triplets(extracted_triplets)
prepared_triplets = MemPipeline.match_triplets_by_id(extracted_triplets, ids)
kg_model.embeddings_db.add_triplets(prepared_triplets)

In [ ]:
kg_model.graph_db.close()